In [ ]:
import json, sys
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import recovery as R

# Reruns the Runs of notebook 10 that two apir-adapter fixes change, with the Run definitions copied from
# notebook 10's plan.jsonl. Notebook 11 takes these Runs from here and every other Run from notebook 10.
#   S3 under wrong_record: a long-lived second client kept upstream's query state (each LWE instance's copy of
#     the public matrix A, about 1.9 GB) alive after the first faulted answer. Every later query of the main
#     client then took about 60 % longer, honest ones included (notebook 11 at commit 29bcc2a, and the
#     manifest's peak footprint, 9.7 GiB against 6.1 GiB).
#   M1 under switch_replica: recovery_s summed the two servers' answers, where the first attempt's latency
#     counts the slower one (spec latency_e2e), so a recovery looked about twice as long as a first attempt.
DAY = repo / "results/raw/rq4-day-20261111"
day = [R.Run(**json.loads(l)) for l in (DAY / "plan.jsonl").read_text().splitlines()]

def changed(r):
    return ((r.arm == "S3" and r.fault == "wrong_record" and r.fault_prob > 0)
            or (r.arm == "M1" and r.policy == "switch_replica"))

rerun = [r for r in day if changed(r)]
day_wall = {json.loads(l)["name"]: json.loads(l)["wall_s"] for l in (DAY / "manifest.jsonl").read_text().splitlines()}
print(f"{len(rerun)} of {len(day)} Runs rerun; they took {sum(day_wall[r.name] for r in rerun) / 60:.0f} min in notebook 10")
for r in rerun:
    print(f"  {r.name}  ({day_wall[r.name]:.0f} s)")

In [ ]:
# About an hour of adapter runs, serially, each under caffeinate, the 14 GiB cap and a hygiene watch.
# Rerunning this cell resumes the same run id.
cfg = R.load_config(R.DEFAULT_CONFIG)
RUN_ID = "rq4-rerun-20261111"
ex = R.prepare(rerun, cfg, R.DEFAULT_CONFIG, run_id=RUN_ID)
plan_file = ex.out_dir / "plan.jsonl"
if not plan_file.exists():
    plan_file.write_text("".join(json.dumps(asdict(r)) + "\n" for r in rerun))
assert [json.loads(l) for l in plan_file.read_text().splitlines()] == [asdict(r) for r in rerun], "the plan changed"
status = R.run_and_report(rerun, ex)

In [ ]:
rows = [json.loads(l) for l in (ex.out_dir / "manifest.jsonl").read_text().splitlines()]
last = {r["name"]: r for r in rows}                     # a resumed Run's latest attempt
outcomes = Counter(r["outcome"] for r in last.values())
print(f"{len(last)} of {len(rerun)} Runs attempted: {dict(outcomes)}; citable {sum(r['citable'] for r in last.values())}")
for r in last.values():
    print(f"  {r['outcome']:3s} {r['name']}  {r['wall_s']:.0f} s (notebook 10: {day_wall[r['name']]:.0f} s)  "
          f"peak {r['peak_footprint_bytes'] / 2**30:.2f} GiB")
print(f"adapter wall time {sum(r['wall_s'] for r in last.values()) / 3600:.2f} h")